# ACF der Amateurfussball-Daten (Landesligen)

Berechnung, Fit, Binning und Plot der ACF der Groesse `X` aus
`data/acf-ready/landesligen.csv`. Die Logik liegt zentral im Modul **`acf_analysis`**
(`src/acf_analysis/acf.py`) und ist identisch zu der, die auf den simulierten Daten laeuft
(`acf_simulated_data.ipynb`) — hier wird sie nur importiert und aufgerufen.

Datengrundlage: 48 Staffel-Saisons aus fuenf Landesverbaenden (Westfalen, Berlin, Suedwest,
Sachsen-Anhalt, Baden), Spielzeiten 2022/23 bis 2025/26. Die Staffeln haben 15 bis 18 Teams,
die Saisonlaengen sind also heterogen.

## Setup — Modul, Daten, Invarianten

Der Import-Bootstrap ist identisch zu `acf_simulated_data.ipynb`: Projektwurzel ueber
`pyproject.toml` finden, deren `src/` auf den Importpfad legen. So laeuft das Notebook mit
**oder** ohne `pip install -e .`.

Geprueft werden das Spaltenschema und `match_number = 0..n-1` je Team-Saison. Der zweite
Test verifiziert zugleich die Zeilensortierung, auf die sich `compute_acf` verlaesst: die
Funktion liest `match_number` nicht, sondern nutzt die Reihenfolge der Zeilen.

In [ ]:
import sys                                                # fuer sys.path (Import-Bootstrap)
from pathlib import Path                                  # Pfade
import numpy as np                                        # numerische Arrays
import pandas as pd                                       # Datentabellen
import matplotlib.pyplot as plt                           # Plots

# --- acf_analysis reproduzierbar importierbar machen ----------------------
ROOT = Path.cwd()                                         # Startpunkt: Verzeichnis des Notebooks
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent                                    # nach oben laufen bis zur Projektwurzel
if str(ROOT / "src") not in sys.path:                     # src/ nur einmal einhaengen
    sys.path.insert(0, str(ROOT / "src"))

from acf_analysis import compute_acf, fit_acf, bin_acf, plot_acf  # zentrale ACF-Logik (Modul)

# --- Datenquelle ---------------------------------------------------------
# data/acf-ready/ ist per .gitignore ausgeschlossen: echte Spieldaten sind sensibel
# (CLAUDE.md Abschnitt 11) und duerfen nicht ins Repo.
DATA_DIR  = ROOT / "data" / "acf-ready"
DATA_FILE = "landesligen.csv"
long_df   = pd.read_csv(DATA_DIR / DATA_FILE)

# --- Achsenbeschriftungen (Fussball: Delta m zaehlt Spieltage) -----------
XLABEL = r"$\Delta m$"
YLABEL = r"$K(\Delta m)$"

# --- Schema- und Invarianten-Check ---------------------------------------
REQUIRED = ["season_id", "date", "team_id", "opponent_id", "X", "is_home", "match_number"]
assert all(c in long_df.columns for c in REQUIRED), "long_df fehlen Spalten"
for _, g in long_df.groupby(["season_id", "team_id"], sort=False):
    assert np.array_equal(g["match_number"].to_numpy(), np.arange(len(g))), "nicht chronologisch"
assert long_df.groupby("season_id")["X"].mean().abs().max() < 1e-9, "E[X] != 0 pro Saison"
assert not long_df[REQUIRED].isna().any().any(), "NA im long_df"

sizes = long_df.groupby(["season_id", "team_id"]).size()  # Spiele je Team-Saison
print(f"Datensatz     : {DATA_FILE}")
print(f"Partien       : {len(long_df) // 2:,}")
print(f"Saisons       : {long_df['season_id'].nunique()}")
print(f"Team-Saisons  : {len(sizes)}")
print(f"Spiele/Team   : min {sizes.min()}, median {int(sizes.median())}, max {sizes.max()}")
print(f"Var(X)        : {long_df['X'].var():.3f}")
long_df.head()

## Zelle 0 — Diagnose: `N(Delta m)`-Verlauf und Wahl von `n_min`

Gibt den Verlauf der Paarzahlen `N(Delta m)` aus, aus dem die Binning-Schwelle `n_min`
gewaehlt wird. Uebernommen wird `n_min = 9000` aus der Auswertung der simulierten Daten:
dieser Datensatz hat mit `N(1) ~ 23 500` praktisch dieselbe Paarzahl beim kleinsten Lag,
sodass die Punkte beider Auswertungen dasselbe statistische Gewicht tragen.

Die Tabelle zeigt zusaetzlich die Spalte `aktive Team-Saisons` — wie viele Einheiten bei
diesem Lag ueberhaupt noch beitragen.

In [ ]:
acf_df = compute_acf(long_df)                             # ACF ueber alle Team-Saisons

# --- Wie viele Team-Saisons tragen bei welchem Lag noch bei? -------------
# Eine Einheit mit n Spielen liefert Paare fuer die Lags 1..n-1.
aktiv = np.array([(sizes - 1 >= d).sum() for d in acf_df["lag"]])

diag = acf_df[["lag", "N", "K", "sigma"]].copy()
diag["aktive Team-Saisons"] = aktiv
diag["Anteil Einheiten"] = (aktiv / len(sizes)).round(3)
print(f"max Lag: {int(acf_df['lag'].max())}   N(1) = {int(acf_df['N'].iloc[0]):,}"
      f"   (Simulation: 37 / 22.940)\n")
print(diag.to_string(index=False))

# --- ab welchem Delta m traegt die Mehrheit der Staffeln nicht mehr bei? --
# Referenz ist die haeufigste Saisonlaenge (16er-Staffeln, 30 Spiele -> Lags bis 29).
haeufigste_laenge = int(sizes.mode().iloc[0])
MIX_CUT = haeufigste_laenge - 1                           # letzter Lag, den die Mehrheit noch traegt
print(f"\nHaeufigste Saisonlaenge: {haeufigste_laenge} Spiele "
      f"({(sizes == haeufigste_laenge).sum()} Team-Saisons)")
print(f"Ab Delta m > {MIX_CUT} tragen nur noch die laengeren Staffeln bei "
      f"({(sizes - 1 > MIX_CUT).sum()} Team-Saisons).")

## Zelle 1 — ACF, Fit und Binning

Alle drei Schritte kommen aus dem Modul, es wird nichts neu implementiert:

- `compute_acf` — `K(Delta m)`, Paarzahl `N`, gepoolte Summen `S`/`SQ` und der
  Standardfehler `sigma`. Einheit ist die Team-Saison.
- `fit_acf` — gewichteter Least-Square-Fit auf allen individuellen Lags (keine Bins),
  einmal mit freiem `tau` und einmal mit fixem `tau = 7`. Gewichtet wird mit `sigma` und
  `absolute_sigma=False`, also nur relativ: Lags mit wenigen Paaren zaehlen im `chi^2`
  weniger.
- `bin_acf` — greedy-Binning nur fuer die Darstellung.

In [ ]:
N_MIN = 9000                                              # ABSOLUT, wie bei den simulierten Daten

fit_res = fit_acf(acf_df)                                 # beide Fits: freies tau + fixes tau=7
bins_df = bin_acf(acf_df, n_min=N_MIN)                    # Binning NUR fuer die Darstellung

singles = [t[0] for t in bins_df["lags_merged"] if len(t) == 1]
print(f"{len(acf_df)} Lags  ->  {len(bins_df)} Bins  (n_min={N_MIN})")
print(f"Einzel-Lags: {len(singles)} (bis Delta m = {max(singles)})\n")
print(bins_df.to_string(index=False))

## Zelle 2 — Ergebnistabelle

Beide Fits nebeneinander. Die normierten Spalten `a/Var(X)` und `b/Var(X)` machen das
Ergebnis mit anderen Datensaetzen vergleichbar, deren `X` eine andere Streuung hat — `K`
hat die Einheit `X^2`, absolute Werte sind also nicht direkt uebertragbar.

In [ ]:
tab = pd.DataFrame([
    {"Fit": "frei",    "a": fit_res.a,  "a±": fit_res.a_err,
     "b": fit_res.b,   "b±": fit_res.b_err,
     "tau": fit_res.tau, "tau±": fit_res.tau_err,
     "chi2/dof": fit_res.chi2_red, "dof": fit_res.dof},
    {"Fit": "tau = 7", "a": fit_res.a7, "a±": fit_res.a7_err,
     "b": fit_res.b7,  "b±": fit_res.b7_err,
     "tau": 7.0, "tau±": np.nan,
     "chi2/dof": fit_res.chi2_red7, "dof": fit_res.dof7},
]).round(4)

var_X = float(long_df["X"].var())
tab["a/Var(X)"] = (tab["a"] / var_X).round(4)             # normiert: Anteil der konstanten Komponente
tab["b/Var(X)"] = (tab["b"] / var_X).round(4)             # normiert: Anteil der Formphase

print(f"Var(X) = {var_X:.3f}    corr(b, tau) = {fit_res.corr_b_tau:.3f}\n")
print(tab.to_string(index=False))

print(f"\nPlateau a = {fit_res.a:.3f} entspricht einer Standardabweichung der konstanten "
      f"Teamstaerke von sqrt(a) = {np.sqrt(max(fit_res.a, 0)):.3f} Toren.")

## Zelle 3 — Plot

Gebinnte ACF-Punkte mit Fehlerbalken, darueber die beiden Fit-Kurven aus allen ungebinnten
Lags. Die gepunktete Linie markiert das `Delta m`, ab dem nur noch die laengeren Staffeln
beitragen.

> **Wichtig:** Das Binning dient nur der Darstellung und geht nicht in den Fit ein.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_acf(bins_df, fit_res, primary="free", ax=ax,          # nur der freie Fit, rote Kurve
         xlabel=XLABEL, ylabel=YLABEL)

# ax.axvline(MIX_CUT, color="grey", ls=":", lw=1.2)         # ab hier fehlen die kurzen Staffeln
# ax.annotate("ab hier tragen nur noch\ndie laengeren Staffeln bei",
#             xy=(MIX_CUT, 1.0), xycoords=("data", "axes fraction"),
#             xytext=(4, -8), textcoords="offset points",
#             va="top", ha="left", fontsize=7, color="grey")

fig.tight_layout()
plt.show()

## Zelle 4 — Export

Exportiert werden ausschliesslich aggregierte Groessen — `K(Delta m)`, Paarzahlen,
Fit-Parameter. Keine Einzelergebnisse, keine Vereinsnamen. `SAVE = False` setzen, wenn
nichts geschrieben werden soll.

In [ ]:
SAVE = True                                               # auf False setzen, um nichts zu schreiben

RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"

if SAVE:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    acf_path = RESULTS_DIR / "real_acf_amateur_fb_landesligen_2022-2026.csv"
    acf_df[["lag", "K", "N", "sigma"]].to_csv(acf_path, index=False)

    fit_path = RESULTS_DIR / "real_acf_amateur_fb_fitparameter.csv"
    tab.to_csv(fit_path, index=False)

    fig_path = FIGURES_DIR / "acf_amateur_fb.pdf"
    fig.savefig(fig_path, bbox_inches="tight")

    for p in (acf_path, fit_path, fig_path):
        print(f"geschrieben: {p.relative_to(ROOT)}")
else:
    print("SAVE = False -- nichts geschrieben.")